<a href="https://colab.research.google.com/github/Arda-Avci/AI-Publisher/blob/main/Google_Colab_AI_Publisher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 AI Publisher - Otonom Video Üretim ve Pazarlama Sunucusu

Bu notebook, **AI Publisher** projesi için Google Colab üzerinde çalışan GPU destekli yapay zekâ sunucusunu (CogVideoX-5b, XTTS-v2, AudioLDM2, GFPGAN/RealESRGAN ve Wav2Lip dudak senkronizasyonu) kurar ve başlatır.

### 💡 Kurulum ve Başlatma Talimatı:
1. Üst menüden **Düzenle (Edit) > Defter Ayarları (Notebook Settings)** seçeneğine gidin.
2. Donanım ivmelendiriciyi **T4 GPU** (veya daha üstü bir ekran kartı) olarak seçip kaydedin.
3. Aşağıdaki kod hücresini çalıştırın. Bağımlılıklar ilk kez kurulurken kernel otomatik olarak yeniden başlatılacaktır (Oturum çöktü uyarısı normaldir).
4. Kernel otomatik kapandıktan sonra **aynı hücreyi ikinci kez çalıştırın**.
5. Ngrok Auth Token istendiğinde ngrok.com adresinden alacağınız token'ınızı girin.
6. Sunucu başarıyla ayağa kalktığında size verilen tünel URL'sini kopyalayıp yerel Node.js projenizin `.env` dosyasındaki `COLAB_URL` alanına yapıştırın.

In [ ]:
GITHUB_TOKEN = "" #@param {type:"string"}
NGROK_TOKEN = "" #@param {type:"string"}

import os, subprocess, shutil, sys
from google.colab import userdata

repo_dir = "/content/AI-Publisher"

if os.path.exists(repo_dir):
    print("Deleting existing repository directory...")
    shutil.rmtree(repo_dir, ignore_errors=True)

if not GITHUB_TOKEN:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    GITHUB_TOKEN = input("GitHub token: ").strip()

if not NGROK_TOKEN:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
if not NGROK_TOKEN:
    NGROK_TOKEN = input("Ngrok Auth Token: ").strip()

repo_url = f"https://{GITHUB_TOKEN}@github.com/Arda-Avci/AI-Publisher.git"

print("Repository klonlanıyor...")
subprocess.run(["git", "clone", repo_url, repo_dir], check=True)

print("colab_setup.py kopyalanıyor...")
shutil.copy(os.path.join(repo_dir, "colab_setup.py"), "/content/colab_setup.py")
print("colab_server.py kopyalanıyor...")
shutil.copy(os.path.join(repo_dir, "colab_server.py"), "/content/colab_server.py")

print("Kurulum betiği canlı loglar eşliğinde çalıştırılıyor...")
try:
    env = os.environ.copy()
    env["GITHUB_TOKEN"] = GITHUB_TOKEN
    env["NGROK_TOKEN"] = NGROK_TOKEN
    
    process = subprocess.Popen(
        ["python3", "-u", "/content/colab_setup.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    # Canlı logları yazdır
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    process.wait()
    
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, process.args)
except subprocess.CalledProcessError as e:
    if e.returncode in [9, -9, 137, 100]:
        print(f"\n[INFO] Kurulum betiği sonlandı (Çıkış Kodu: {e.returncode}). Oturum yenileniyor...")
        import os
        os.kill(os.getpid(), 9)
    else:
        print(f"\n❌ Kurulum hatayla bitti! Çıkış Kodu: {e.returncode}")
        raise e

## 🛠️ Seçenek C: Tüm Docker İmajlarını İnşa Et ve Google Drive'a Kaydet (Maliyet Tasarruflu CPU Modu)

Bu bölüm, AI Publisher projesinde kullanılan 11 Docker konteyner imajını Google Colab **CPU** çalışma zamanında sıfırdan inşa eder, `pigz` kullanarak çok çekirdekli paralel sıkıştırır, Google Drive'a yükler ve bütünlük kontrolüyle doğrular.

### 💡 Maliyet Tasarrufu ve Çalıştırma Talimatı:
1. Üst menüden **Düzenle (Edit) > Defter Ayarları (Notebook Settings)** seçeneğine gidin.
2. Donanım ivmelendiriciyi **CPU** (GPU yok) olarak seçip kaydedin (CPU modu ücretsizdir veya çok daha ucuzdur).
3. Aşağıdaki hücreyi çalıştırın. Hücre, tüm imajları sırayla derleyecek, Drive'a `.tar.gz` olarak kaydedecek ve bütünlüklerini doğrulayacaktır.
4. **Otomatik Kapanma (Unassign):** Derleme ve doğrulama tamamlandığında, `AUTO_DISCONNECT = True` olarak ayarlandıysa, Colab oturumu otomatik olarak sonlandırılacaktır. Bu sayede işlem bittiğinde ek süre için ücretlendirilmezsiniz.

In [ ]:
GITHUB_TOKEN = "" #@param {type:"string"}
AUTO_DISCONNECT = True #@param {type:"boolean"}

import os, subprocess, shutil, sys
from google.colab import userdata

repo_dir = "/content/AI-Publisher"
drive_dir = "/content/drive/MyDrive/Colab Notebooks/docker/images"

# 1. Google Drive Bağlantısı
print("📥 Google Drive bağlanıyor...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive başarıyla bağlandı!")
except Exception as e:
    print(f"❌ Google Drive bağlanamadı: {e}")
    sys.exit(1)

print("==========================================")
print("⚠️ Google Colab Kısıtlamaları Nedeniyle Derleme Yerelleştirildi!")
print("==========================================")
print("Google Colab'ın güncellenen kernel ve cgroup kısıtlamaları (read-only filesystem)")
print("nedeniyle Docker/Podman derlemeleri Colab üzerinde teknik olarak engellenmektedir.")
print("Bu yüzden kredilerinizin boşa gitmesini önlemek amacıyla derleme süreci tamamen")
print("yerel bilgisayarınıza taşınmıştır.")
print("")
print("👉 ADIM ADIM YEREL DERLEME TALİMATI:")
print("1. Yerel bilgisayarınızda Docker Desktop uygulamasının açık ve çalışır olduğundan emin olun.")
print("2. Windows PowerShell terminalini açıp proje dizinine gidin:")
print("   cd C:\\Users\\Damla\\Proje\\AI-Publisher")
print("3. colab_docker/ klasöründeki yerel derleme betiğini çalıştırın:")
print("   .\\colab_docker\\build_local.ps1")
print("4. Bu betik sırayla tüm model imajlarını yerelde derleyecek, sıkıştıracak ve")
print("   'colab_docker/dist/' klasörüne kaydedecektir (Colab kredilerinizden 0 harcama!).")
print("5. Derleme bittiğinde, 'colab_docker/dist/' içindeki 11 adet '.tar.gz' dosyasını")
print("   Google Drive'ınızdaki 'Colab Notebooks/docker/images/' dizinine sürükleyip bırakarak yükleyin.")
print("6. Yükleme tamamlandıktan sonra Colab sunucusunu (1. Hücre) normal şekilde başlatabilirsiniz.")
print("==========================================")
